# 06 — Inference with your distilled model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/06_distilled_inference.ipynb)

You trained a student in notebook 05. Now **use it every day in production**. This notebook walks through four serving paths — from "load the raw LoRA in a Python script" to "swap it behind a routing alias so your app code never changes" — and closes with how to tell when to retrain.

In this notebook:

1. **Load the student** with Transformers + PEFT — smallest possible serving
2. **Serve via llama.cpp** (GGUF) — CPU-only, edge-ready
3. **Register in the OpenTracy engine** — behind a routing alias
4. **The alias swap** — the closing move of the pipeline
5. **Monitor drift** — when to retrain

## Prerequisites

Pick one:

- **You ran notebook 05** and have `artifacts["adapter_path"]` and `artifacts["gguf_paths"]` from that session — plug them in below.
- **You want to see the flow without training** — we provide a demo-adapter HuggingFace repo you can download.

> **Runtime:** CPU is enough for everything in this notebook. The student runs on CPU once quantized.

## Setup

In [ ]:
!pip install opentracy transformers peft accelerate llama-cpp-python -q

In [ ]:
# Point these at the outputs from notebook 05 — or use the demo values to skip training
ADAPTER_PATH = "./demo-student/adapter"        # from notebook 05's artifacts["adapter_path"]
GGUF_PATH    = "./demo-student/model.q4_k_m.gguf"  # from artifacts["gguf_paths"]["q4_k_m"]
BASE_MODEL   = "unsloth/Llama-3.2-1B-Instruct"

import os, pathlib
if not pathlib.Path(ADAPTER_PATH).exists():
    print("No local adapter found — downloading the demo student from HuggingFace...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="OpenTracy/demo-ticket-classifier-v1",
        local_dir="./demo-student",
    )
    print("demo adapter ready")
else:
    print("using your trained adapter at", ADAPTER_PATH)

## 1. Serving path A — Transformers + PEFT

The most basic Python path: load the base model, attach the LoRA adapter, run inference. Works anywhere Transformers works (FastAPI, Celery worker, cron script, notebook).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
student = PeftModel.from_pretrained(base, ADAPTER_PATH)
student.eval()
print("student loaded")

In [ ]:
def ask_student(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(student.device)
    with torch.no_grad():
        out = student.generate(inputs, max_new_tokens=10, temperature=0, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

for ticket in [
    "My invoice for March hasn't arrived",
    "Getting 500 errors when calling /v1/users/me",
    "Please add a dark mode",
]:
    prompt = f"Classify this ticket (billing/technical/feature_request/other): '{ticket}'"
    print(f"[{ask_student(prompt):<18}]  {ticket}")

This is your student running. Zero API calls, zero egress cost, everything local.

## 2. Serving path B — llama.cpp (GGUF, CPU-only)

For edge deployments, on-device inference, or when you don't want Python in the serving path, use the GGUF file produced by notebook 05.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=GGUF_PATH,
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)
print("GGUF model loaded")

In [ ]:
def ask_student_gguf(prompt: str) -> str:
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0,
    )
    return resp["choices"][0]["message"]["content"].strip()

for ticket in [
    "How do I update my billing address?",
    "Rate limit header says 60/min but we're being throttled at 30",
]:
    prompt = f"Classify this ticket (billing/technical/feature_request/other): '{ticket}'"
    print(f"[{ask_student_gguf(prompt):<18}]  {ticket}")

The 4-bit GGUF file is ~500 MB and runs on any CPU. Common deployment targets:

- **llama.cpp server** — `./server -m model.q4_k_m.gguf -c 2048`
- **Ollama** — `ollama create ticket-classifier -f Modelfile`
- **LM Studio** — drop the file in the models folder
- **On-device mobile** — llama.cpp compiles on iOS / Android

## 3. Production path — register in the OpenTracy engine

In production you don't want your app aware of "adapter path" or "GGUF file". You want it to call `model="ticket-classifier"` and forget about the rest. The engine holds that mapping.

In [ ]:
import subprocess, time, httpx

# Management API
api = subprocess.Popen(
    ["uvicorn", "opentracy.api.server:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(30):
    try:
        httpx.get("http://127.0.0.1:8000/health", timeout=2).raise_for_status()
        print("management API up")
        break
    except Exception:
        time.sleep(1)

In [ ]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard at http://<host>:3000/ — uncomment the two
# lines below BEFORE the `import opentracy` call in the next cell.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

# import os
# os.environ["OPENTRACY_ENGINE_URL"] = "http://localhost:8080"  # or your remote host


In [ ]:
from opentracy import Distiller

d = Distiller(base_url="http://127.0.0.1:8000")

d.register_model(
    id="ticket-classifier",
    adapter_path=ADAPTER_PATH,
    gguf_path=GGUF_PATH,
)
print("registered — alias 'ticket-classifier' now routes to your student")

In [ ]:
import opentracy as ot

resp = ot.completion(
    model="ticket-classifier",    # the alias
    messages=[{"role": "user", "content":
        "Classify this ticket (billing/technical/feature_request/other): "
        "'Cancel my subscription'"}],
    max_tokens=10,
    force_engine=True,
)
print("answer:", resp.choices[0].message.content.strip())
print(f"cost:   ${resp._cost:.8f}  (served locally — no provider fee)")

## 4. The alias swap — the closing move of the pipeline

This is the piece people don't see coming. Suppose your app has been shipping for weeks with:

```python
resp = ot.completion(model="smart", messages=[...])
```

And `smart` has been aliased to `openai/gpt-4o`. When the student is ready, you re-point the alias:

In [ ]:
# Before: smart → gpt-4o
d.set_alias("smart", target="openai/gpt-4o")
print("Before swap:")
resp = ot.completion(
    model="smart",
    messages=[{"role": "user", "content": "Classify: 'Please refund the double charge'"}],
    max_tokens=10,
    force_engine=True,
)
print(f"  answered by: {resp._routing.get('selected_model')}  cost=${resp._cost:.6f}")

In [ ]:
# After: smart → ticket-classifier
d.set_alias("smart", target="ticket-classifier")
print("After swap:")
resp = ot.completion(
    model="smart",
    messages=[{"role": "user", "content": "Classify: 'Please refund the double charge'"}],
    max_tokens=10,
    force_engine=True,
)
print(f"  answered by: {resp._routing.get('selected_model')}  cost=${resp._cost:.8f}")

**Your app code didn't change.** The same `model="smart"` string went from an expensive frontier call to a local 4-bit student. That's the moment cost savings actually land in your invoice.

### Rolling back

If the student starts mis-classifying, point the alias back — the engine picks up the change without restart:

```python
d.set_alias("smart", target="openai/gpt-4o")   # instant rollback
```

## 5. Monitor drift — when to retrain

Traffic shifts. The benchmark clusters you trained on may not match next quarter's prompts. Two signals to watch:

1. **Student-vs-teacher agreement on new traffic.** Sample N% of requests, send the same prompt to both, compare labels.
2. **Cluster distribution shift.** If your traffic moves into clusters the student wasn't trained for, the router's error estimates will climb.

In [ ]:
# Shadow 10 recent tickets through both models, compare
shadow_tickets = [
    "Credit card declined on renewal — please help",
    "Need webhook delivery retries with exponential backoff",
    "Add a cron-based scheduled export",
    "What region is your data stored in?",
    "Account locked out after failed 2FA",
    "Please upgrade my plan to annual",
    "GraphQL endpoint returning 502 intermittently",
    "Invoice PDF missing line items",
    "Support OAuth device flow?",
    "When is the next product webinar?",
]

agree = 0
student_cost_total = 0.0
teacher_cost_total = 0.0

for t in shadow_tickets:
    prompt = f"Classify this ticket (billing/technical/feature_request/other): '{t}'"

    s = ot.completion(model="ticket-classifier", messages=[{"role":"user","content":prompt}],
                      max_tokens=10, force_engine=True)
    t_resp = ot.completion(model="openai/gpt-4o", messages=[{"role":"user","content":prompt}],
                           max_tokens=10, temperature=0)

    s_label = s.choices[0].message.content.strip().lower()
    t_label = t_resp.choices[0].message.content.strip().lower()
    if s_label == t_label:
        agree += 1
    student_cost_total += s._cost
    teacher_cost_total += t_resp._cost

print(f"agreement:       {agree}/{len(shadow_tickets)}  ({100*agree/len(shadow_tickets):.0f}%)")
print(f"student cost:    ${student_cost_total:.6f}")
print(f"teacher cost:    ${teacher_cost_total:.6f}")
print(f"savings:         {100*(teacher_cost_total - student_cost_total)/teacher_cost_total:.0f}%")

**Rules of thumb:**

| Agreement | Action |
| --- | --- |
| ≥ 95% | Stay on student |
| 90–95% | Monitor; review the disagreements manually |
| 80–90% | Shadow the teacher in parallel; plan a retrain |
| < 80% | Point the alias back to the teacher; retrain with recent traces |

Shadow sampling and drift dashboards are built into the self-hosted UI under **Evaluations → Drift**.

## Clean up

In [ ]:
api.terminate()
api.wait(timeout=5)
print("api shut down")

## The full loop, in review

You've now done every step of the OpenTracy pipeline:

1. **Quickstart** — cost/latency on every call (notebook 01)
2. **Drop-in** — zero-code OpenAI migration (notebook 02)
3. **Routing** — auto-pick a model per prompt (notebook 03)
4. **End-to-end** — measure savings on a real workload (notebook 04)
5. **Distillation** — train a custom student from your traffic (notebook 05)
6. **Inference & alias swap** — ship it (**this notebook**)

The value compounds: every cluster you distill shaves more off the cost curve, and the alias swap means the app never learns about any of it.

## Further reading

- [Self-host guide](https://docs.opentracy.ai/guides/self-host) — Docker Compose for the full stack (ClickHouse, UI, management API)
- [Auto-routing concepts](https://docs.opentracy.ai/concepts/auto-routing) — how cluster assignment + error profiles work under the hood
- [Router reference](https://docs.opentracy.ai/api-reference/router) — the explicit `Router` class for rule-based routing on top of learned routing